In [ ]:
# model

import math
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=4, max_len=256):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=0.0,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, size, device):
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float("-inf"))
        return mask

    def forward(self, x):
        _, t = x.size()
        mask = self.generate_square_subsequent_mask(t, x.device)
        x = self.token_embedding(x)
        x = self.pos_encoding(x)
        x = self.transformer(x, mask=mask)
        return self.fc_out(x)


In [ ]:
# temperature sampling + metrics

import json
import pickle
import re
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_CONTEXT = 256
PROMPT = "First Citizen: Before we proceed any further, hear me speak."
MAX_LENGTH = 500
TEMPERATURES = [0.7, 1.0, 1.3]
RUNS_PER_TEMP = 5


def load_vocab():
    with open("char_to_idx.pkl", "rb") as f:
        char_to_idx = pickle.load(f)
    with open("idx_to_char.pkl", "rb") as f:
        idx_to_char = pickle.load(f)
    return char_to_idx, idx_to_char


def temperature_decode(model, start_text, char_to_idx, idx_to_char, temperature=1.0, max_length=500):
    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    missing = [ch for ch in start_text if ch not in char_to_idx]
    if missing:
        raise ValueError(f"prompt contains characters outside vocabulary: {sorted(set(missing))}")

    model.eval()
    generated = [char_to_idx[ch] for ch in start_text]
    input_ids = torch.tensor([generated], dtype=torch.long, device=DEVICE)

    for _ in range(max_length):
        with torch.no_grad():
            logits = model(input_ids)
            next_logits = logits[0, -1] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

        generated.append(next_token)
        input_ids = torch.tensor([generated[-MAX_CONTEXT:]], dtype=torch.long, device=DEVICE)

    return "".join(idx_to_char[i] for i in generated)


def compute_perplexity(model, data_tensor, vocab_size):
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0

    for seq in data_tensor:
        x = seq[:-1].unsqueeze(0).to(DEVICE)
        y = seq[1:].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        total_loss += loss.item()

    avg_loss = total_loss / len(data_tensor)
    ppl = torch.exp(torch.tensor(avg_loss)).item()
    return avg_loss, ppl


def compute_ttr(text):
    tokens = list(text)
    return len(set(tokens)) / len(tokens) if tokens else 0.0


def shakespeare_line_score(text):
    lines = text.split("\n")
    valid_lines = 0
    for line in lines:
        words = line.strip().split()
        if 5 <= len(words) <= 12:
            valid_lines += 1
    return valid_lines / max(len(lines), 1) * 100


def levenshtein_distance(a, b):
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            curr.append(min(
                prev[j] + 1,
                curr[j - 1] + 1,
                prev[j - 1] + cost,
            ))
        prev = curr
    return prev[-1]


def compute_cer(pred_text, target_text):
    return levenshtein_distance(pred_text, target_text) / max(len(target_text), 1)


In [ ]:
# run evaluation

char_to_idx, idx_to_char = load_vocab()
vocab_size = len(char_to_idx)

model = DecoderOnlyTransformer(vocab_size).to(DEVICE)
model.load_state_dict(torch.load("greedy_model.pth", map_location=DEVICE))
model.eval()

val_data = np.load("val.npy")
val_tensor = torch.tensor(val_data, dtype=torch.long)
val_text = "".join(idx_to_char[int(i)] for seq in val_tensor for i in seq)

val_loss, val_ppl = compute_perplexity(model, val_tensor, vocab_size)
print(f"Validation Loss: {val_loss:.4f} | Perplexity: {val_ppl:.3f}")

results = []
for temp in TEMPERATURES:
    ttr_scores = []
    line_scores = []
    cer_scores = []
    generated_samples = []

    for _ in range(RUNS_PER_TEMP):
        text = temperature_decode(
            model,
            PROMPT,
            char_to_idx,
            idx_to_char,
            temperature=temp,
            max_length=MAX_LENGTH,
        )
        generated_samples.append(text)
        ttr_scores.append(compute_ttr(text))
        line_scores.append(shakespeare_line_score(text))
        cer_scores.append(compute_cer(text[: len(val_text)], val_text[: len(text)]))

    row = {
        "temperature": temp,
        "val_loss": round(val_loss, 4),
        "val_ppl": round(val_ppl, 4),
        "ttr_mean": round(float(np.mean(ttr_scores)), 4),
        "line_score_mean": round(float(np.mean(line_scores)), 4),
        "cer_mean": round(float(np.mean(cer_scores)), 4),
        "sample_text": generated_samples[0],
    }
    results.append(row)

    print(f"\nTemperature={temp:.1f}")
    print(f"TTR (mean): {row['ttr_mean']:.4f}")
    print(f"Shakespearean Line Score (mean): {row['line_score_mean']:.2f}%")
    print(f"CER (mean): {row['cer_mean']:.4f}")
    print("Sample Generated Text:")
    print(generated_samples[0])

Path("temperature_scaling_report.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print("\nSaved report to temperature_scaling_report.json")
